# 06.4 — Positional Encoding

**Problem it solves:** Self-attention has no inherent sense of order. 'The cat bit the dog' and 'The dog bit the cat' have the same attention scores if positions aren't encoded.

**Two approaches:**
1. **Sinusoidal (Vaswani 2017):** Fixed, deterministic. PE(pos, 2i) = sin(pos / 10000^{2i/d_model})
2. **Learned:** Train a position embedding table (BERT uses this)

---

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import math

class SinusoidalPositionalEncoding(nn.Module):
    """
    Fixed sinusoidal positional encoding from 'Attention Is All You Need'.
    
    PE(pos, 2i)   = sin(pos / 10000^{2i/d_model})
    PE(pos, 2i+1) = cos(pos / 10000^{2i/d_model})
    
    Different frequencies: low dimensions = high frequency (fine-grained)
                           high dimensions = low frequency (coarse position)
    This lets the model learn to attend by relative position:
    PE(pos+k) is a linear function of PE(pos) for any fixed offset k.
    """
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Compute PE table: [max_len, d_model]
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # [max_len, 1]
        
        # Frequencies: 1/10000^{2i/d_model} for i=0,1,...,d_model/2
        # Equivalent to exp(-log(10000) * 2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)  # even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dims
        
        # Register as buffer (not a parameter, but saved with model)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [batch, seq_len, d_model]"""
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class LearnedPositionalEncoding(nn.Module):
    """Trainable position embeddings (used in BERT, GPT)."""
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)
        nn.init.normal_(self.pe.weight, std=0.02)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device)
        return x + self.pe(positions)

# Visualize sinusoidal PE
d_model = 64
max_len = 50
pe_module = SinusoidalPositionalEncoding(d_model=d_model, max_len=max_len)
pe_matrix = pe_module.pe[0].numpy()  # [max_len, d_model]

print(f"PE matrix shape: {pe_matrix.shape}")
print(f"PE range: [{pe_matrix.min():.3f}, {pe_matrix.max():.3f}]")

# Show similarity between positions
def pe_similarity(pe, pos1, pos2):
    v1, v2 = pe[pos1], pe[pos2]
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

print("\nCosine similarity between position pairs:")
pairs = [(0,1), (0,5), (0,10), (0,25), (0,49), (10,11), (10,15)]
for p1, p2 in pairs:
    sim = pe_similarity(pe_matrix, p1, p2)
    print(f"  pos {p1:3d} vs pos {p2:3d}: {sim:.4f}")

print()
print("Nearby positions are similar, distant positions are less similar.")
print("This gives the model a continuous sense of 'closeness'.")

In [ ]:
# Why sinusoidal? The relative position property.
# PE(pos+k) = R_k * PE(pos) for a fixed rotation matrix R_k
# The model can learn to compute relative positions from this.

print("Relative position property of sinusoidal encoding:")
print()
print("PE(pos+k) can be computed as a LINEAR FUNCTION of PE(pos).")
print("This means the model can represent 'k positions ahead' as a matrix multiply.")
print()

# Demonstrate: for 2D case
def pe_2d(pos, freq=1.0):
    return np.array([np.sin(pos * freq), np.cos(pos * freq)])

# Rotation matrix for offset k
def rotation_2d(k, freq=1.0):
    return np.array([
        [np.cos(k * freq), np.sin(k * freq)],
        [-np.sin(k * freq), np.cos(k * freq)]
    ])

pos = 5
k = 3
freq = 1.0

direct = pe_2d(pos + k, freq)
via_rotation = rotation_2d(k, freq) @ pe_2d(pos, freq)

print(f"PE(pos={pos}) = {pe_2d(pos, freq).round(4)}")
print(f"PE(pos+{k}={pos+k}) = {direct.round(4)}")
print(f"R_{k} @ PE({pos}) = {via_rotation.round(4)}")
print(f"Equal: {np.allclose(direct, via_rotation)}")

print()
print("RoPE (Rotary Position Embedding, 2021) exploits this more directly")
print("and is now used in most modern LLMs (LLaMA, Mistral, Qwen, etc.)")

In [ ]:
# Sinusoidal vs Learned: practical comparison

print("Sinusoidal vs Learned PE:")
print()
print("Sinusoidal:")
print("  + No extra parameters")
print("  + Generalizes to sequences LONGER than training length")
print("    (PE(10000) is defined even if max training was 512)")
print("  - Can't adapt to task-specific position patterns")
print()
print("Learned (BERT, GPT-2):")
print("  + Can learn task-optimal patterns")
print("  - Extra parameters (max_len × d_model = 512 × 768 = 393k for BERT)")
print("  - Doesn't generalize beyond training length")
print()
print("Modern (RoPE, ALiBi):")
print("  + Better length generalization than both")
print("  + RoPE: apply rotation to Q and K before dot product")
print("  + ALiBi: add position bias to attention scores directly")
print("  These are now the standard in most 2023+ LLMs")

## Summary

**Position encoding** injects order information into a position-unaware architecture.

**Key insight:** The sinusoidal frequencies cover all timescales — from single-position to thousand-position periodicity — giving the model a multi-resolution view of position.

---
**Next:** 06.5 — Full Transformer Encoder